# 06 — The models (stage 9): the runner, XGB-dev and the neural models in development

**Purpose.** One runner (`python -m solarpv.pipeline`) trains every model of the study in
sequence, evaluates them with the evaluation layer of stage 8, explains them and writes one
report per run. This notebook shows the two development runs of stage 9, XGB-dev (W2, rerun in
W5 with the new steps) and the five neural models (W4), with what W5 added: the verified input
(`select`), the explanations (`explain`) and the run report (`report`).

**Rules.**
- Development runs read train and validation only, not one test row.
- Validation chose the hyper-parameters and the stopping rounds (D-10), so every number here is
  a tuning-split number, optimistic by construction. The final run (stage 10) and the test split
  (stage 11) come once, after gate G-1, on frozen code.
- Every model and every reference is scored on the same 2,901 daytime validation hours per
  horizon.

**Runs.** `outputs/runs/20261112T063719Z_dev` (XGB) and `outputs/runs/20261111T160714Z_dev` (BiLSTM, TFT-96, TFT-168, TFT baseline, PatchTST).

**Documented in** `docs/stage9_plan.md` §13–§19, OP-49 to OP-58, run journal R9-01 to R9-21,
tests T-51 to T-91, `docs/runbook_final.md`.

## 1. The runner and its steps

In [1]:
import json

import pandas as pd

from solarpv.config.paths import REPO_ROOT
from solarpv.pipeline.config import STEPS
from solarpv.pipeline.runner import select_steps

XD = REPO_ROOT / "outputs/runs/20261112T063719Z_dev"  # XGB-dev, with select, explain, report
ND = REPO_ROOT / "outputs/runs/20261111T160714Z_dev"  # the neural models, resumed with them
print("default sequence:", select_steps())
pd.DataFrame([(s.name, s.stage, s.device, s.purpose) for s in STEPS],
             columns=["step", "stage", "device", "purpose"])

default sequence: ['select', 'baselines', 'xgb', 'bilstm', 'tft96', 'tft168', 'tftbase', 'patchtst', 'evaluate', 'explain', 'report']


,step,stage,device,purpose
0,select,7,CPU,verifies the stage-7 selection and records its...
1,baselines,8,CPU,"persistence, climatology, clear sky x MEAN_DAY..."
2,xgb,9/10,CPU,XGB-Opt
3,bilstm,9/10,GPU,BiLSTM-Opt
4,tft96,9/10,GPU,TFT with a 96-hour encoder
5,tft168,9/10,GPU,TFT with a 168-hour encoder
6,tftbase,9/10,GPU,TFT with fixed hyper-parameters
7,patchtst,9/10,GPU or CPU,"PatchTST, univariate"
8,evaluate,8/11,CPU,the evaluation layer on every prediction file ...
9,explain,11,CPU,"SHAP (XGB), TFT attention and variable selecti..."


**What it shows.** The 12 steps of the runner in their order. A run executes a
selection of them (`--only`, `--from`, `--to`), and `--resume` skips a step whose fingerprint
(mode, configuration, code commit, input hashes) is unchanged.

**What it means.** The last step, `test`, is not in the default sequence: it is the second
phase of a final run and runs only with `--allow-test`, after the stage-10 acceptance has been
written (D-32, X51). In development the test split cannot be reached at all.

## 2. The input of every model, verified (`select`, D-30)

In [2]:
sel = json.loads((XD / "select" / "select.json").read_text(
    encoding="utf-8"))
print("model_inputs.json", sel["model_inputs_sha256"][:16], "| rows read:", sel["rows_read"])
pd.DataFrame(sel["inputs"])

model_inputs.json b693622407f2246e | rows read: 0


,key,source,n_features,matches
0,indirect_D1,05e,31,True
1,indirect_D2,05e,28,True
2,indirect_D3,05e,32,True
3,direct_D1,full pool,96,True
4,direct_D2,full pool,89,True
5,direct_D3,full pool,82,True


**What it shows.** The `select` step reruns nothing (stage 7 is closed, R7-04 and R7-05).
It checks that the stage-7 outputs are unmodified, that the tables named in the stage-7 run logs
are the current ones, that the selection read train only with one validation look, and that
`model_inputs.json` equals the registered rule applied to the committed checks.

**What it means.** The indirect route uses the 05e sets of 31,
28 and 32 features, the
direct route the full pool of 96,
89 and 82 (F-29). Both runs
verified the same file (SHA-256 `b693622407f2246e`), and every model step of a
run refuses another one (M25).

## 3. XGB-dev against the references

In [3]:
bench = pd.read_csv(XD / "evaluate" / "val" / "benchmark_summary.csv")
bench.pivot_table(index=["target", "model"], columns="horizon", values="nRMSE_kW_pct").round(2)

horizon                  1      2      3
target   model                          
direct   clearsky_k   8.96   8.96   8.96
         climatology  8.04   8.04   8.04
         persistence  9.68  10.82  10.72
         xgb_sota     6.66   6.81   6.43
indirect clearsky_k   8.96   8.96   8.96
         climatology  8.06   8.06   8.06
         persistence  9.68  10.84  10.76
         xgb_sota     6.69   6.91   6.55

**What it shows.** Validation nRMSE in % of the 3,449 kW capacity (the indirect route
converted to kW): XGB-dev (20 trials, seed 42) against the three references.

**What it means.** XGB-dev scores 6.69 / 6.91 /
6.55 % (indirect) and 6.66 / 6.81 /
6.43 % (direct) at D+1 / D+2 / D+3, below every reference at every horizon
(climatology 8.04 %, persistence 9.68 % at direct D+1).
This run reproduces the XGB dev run of W2 (`outputs/runs/20261107T135938Z_dev`) exactly: the largest difference over
every model, target, horizon and unit is 0.0. D+3 below D+1 is the validation-sample
pattern examined in R9-06.

In [4]:
dm = pd.read_csv(XD / "evaluate" / "val" / "dm_tests_kw.csv")
dm[dm["model_b"] == "xgb_sota"][["target", "horizon", "model_a", "p_value_hln", "pct_of_mde",
                                  "significant"]].round(4)

,target,horizon,model_a,p_value_hln,pct_of_mde,significant
0,direct,1,clearsky_k,0.0000,156.5945,True
1,direct,1,climatology,0.0008,103.2991,True
2,direct,1,persistence,0.0000,138.9661,True
3,direct,2,clearsky_k,0.0000,149.7470,True
4,direct,2,climatology,0.0013,94.6305,True
5,direct,2,persistence,0.0000,135.7727,True
6,direct,3,clearsky_k,0.0000,159.3524,True
7,direct,3,climatology,0.0002,109.6998,True
8,direct,3,persistence,0.0000,120.3594,True
9,indirect,1,clearsky_k,0.0000,152.4501,True


**What it shows.** Diebold–Mariano tests in kW (Q-19) of XGB-dev against every reference,
with the HLN correction and Bonferroni.

**What it means.** 18 of 18 comparisons are significant.
Against climatology the effect is 93 to 110
% of the minimum detectable effect: better, with a modest margin (a point of the G-1 list,
R9-11).

## 4. The neural models in development

In [5]:
nb = pd.read_csv(ND / "evaluate" / "val" / "benchmark_summary.csv")
nb.pivot_table(index=["target", "model"], columns="horizon", values="nRMSE_kW_pct").round(2)

horizon                  1      2      3
target   model                          
direct   clearsky_k   8.96   8.96   8.96
         climatology  8.04   8.04   8.04
         lstm_sota    6.75   6.95   6.51
         patchtst     8.09   8.27   8.29
         persistence  9.68  10.82  10.72
         tft          6.87   7.04   6.71
         tft_168      6.79   7.01   6.65
         tft_96       6.75   6.94   6.57
indirect clearsky_k   8.96   8.96   8.96
         climatology  8.06   8.06   8.06
         lstm_sota    6.97   7.17   6.87
         patchtst     8.77   8.90   8.86
         persistence  9.68  10.84  10.76
         tft          6.83   7.10   6.81
         tft_168      6.77   7.05   6.66
         tft_96       6.78   6.96   6.68

**What it shows.** The five neural models of v6 with the dev budgets of D-17 (one third
of the v6 trials, the full epochs, seed 42), on the same 2,901 hours.

**What it means.** 30 of 30 model × target × horizon cases are below
persistence. At D+1 the best model is `xgb_sota`
(6.69 %) in the indirect route and `xgb_sota`
(6.66 %) in the direct route; the tuned neural models are at most
0.32 points above XGB-dev, as the literature expects for well-tuned families (C1).
PatchTST sees only the target series (L-19) and stays above climatology everywhere.

In [6]:
train = pd.read_csv(ND / "report" / "report_training.csv")
train[["step", "subject", "unit_of_best", "best", "cap", "run", "gap_ratio", "value_kw",
       "final_minus_best_trial_pp"]].round(3)

,step,subject,unit_of_best,best,cap,run,gap_ratio,value_kw,final_minus_best_trial_pp
0,bilstm,indirect seed 42,epoch,51,60,60,0.723,7.005,0.161
1,bilstm,direct seed 42,epoch,17,60,27,0.719,6.740,0.000
2,tft96,indirect seed 42,epoch,16,100,26,0.716,6.808,0.000
3,tft96,direct seed 42,epoch,32,100,42,0.677,6.756,0.000
4,tft168,indirect seed 42,epoch,26,100,36,0.726,6.827,0.000
5,tft168,direct seed 42,epoch,41,100,51,0.743,6.819,0.000
6,tftbase,indirect seed 42,epoch,18,150,28,0.741,6.915,NaN
7,tftbase,direct seed 42,epoch,58,150,68,0.771,6.878,NaN
8,patchtst,indirect seed 42,step,4900,5000,4999,NaN,8.844,NaN
9,patchtst,direct seed 42,step,4100,5000,4999,NaN,8.216,NaN


**What it shows.** The training diagnostics of every final fit, from the run report: the
best epoch (PatchTST: step) against its cap, the train/validation gap, and the final fit against
its best trial.

**What it means.** The BiLSTM indirect final fit scores 7.00 % against
6.84 % for its best 25-epoch trial
(+0.16 points): early stopping on the quantile loss does not
follow the kW objective there. This point and the others of §6 go to the G-1 list; nothing is
retuned on validation.

## 5. What the models rely on (`explain`, D-31)

In [7]:
chk = pd.read_csv(XD / "explain" / "xgb" / "xgb_shap_checks.csv")
chk[["target", "horizon", "n_rows", "unit", "n_rounds", "rounds_in_booster",
     "additivity_relative", "contribs_max_diff", "unsliced_additivity_max",
     "dominance_first_over_third"]]

,target,horizon,n_rows,unit,n_rounds,rounds_in_booster,additivity_relative,contribs_max_diff,unsliced_additivity_max,dominance_first_over_third
0,indirect,1,2901,k,341,391,1.361448e-06,0.0,0.010362,2.173395
1,direct,1,2901,kW,46,96,3.901202e-07,0.0,199.443140,6.716369
2,indirect,2,2901,k,30,80,3.572495e-07,0.0,0.113854,2.439497
3,direct,2,2901,kW,213,213,7.007865e-07,0.0,0.002417,3.530114
4,indirect,3,2901,k,93,143,8.267016e-07,0.0,0.056316,1.960114
5,direct,3,2901,kW,32,82,2.755914e-07,0.0,181.263670,3.631162


**What it shows.** Exact TreeSHAP of every XGB booster on the
2,901 scored hours per horizon, with its checks.

**What it means.** The explanation adds up to the forecast within
1.4e-06 of the output scale, and XGBoost's own contributions agree
(largest difference 0). The saved boosters hold 50 rounds more
than the forecast uses; explained whole they would be off by up to
199 (k or kW), so every explanation slices the booster to
its forecast rounds first (X46). No feature dominates by the stage-7 measure (first over third
at most 6.7, threshold 10).

In [8]:
feat = pd.read_csv(XD / "explain" / "xgb" / "xgb_shap_features.csv")
top = feat[(feat["seed"] == 42) & (feat["rank"] <= 3)]
top[["target", "horizon", "rank", "feature", "group", "mean_abs_kw", "mean_kw"]].round(1)

,target,horizon,rank,feature,group,mean_abs_kw,mean_kw
0,indirect,1,1,fcst_gfs_k_phys,nwp_gfs,45.4,-15.4
1,indirect,1,2,fcst_gfs_kt_poa_daymean,nwp_gfs,41.8,18.5
2,indirect,1,3,fcst_gfs_lcc,nwp_gfs,20.9,5.7
31,direct,1,1,fcst_gfs_poa,nwp_gfs,455.9,317.1
32,direct,1,2,fcst_gfs_pv_kw,nwp_gfs,385.8,301.8
33,direct,1,3,poa_global_clearsky,clearsky,67.9,66.6
127,indirect,2,1,fcst_gfs_kt_poa_daymean,nwp_gfs,48.1,20.5
128,indirect,2,2,fcst_gfs_k_phys,nwp_gfs,36.1,-11.9
129,indirect,2,3,fcst_gfs_lcc,nwp_gfs,19.7,2.6
155,direct,2,1,fcst_gfs_poa,nwp_gfs,443.5,321.0


**What it shows.** The three largest mean |SHAP| (kW) per target and horizon, and their
mean signed value.

**What it means.** A GFS forecast is the first feature everywhere: at D+1
`fcst_gfs_k_phys` (indirect) and `fcst_gfs_poa` (direct). The
GFS group holds 56 to 79 % of the group attribution (the mean over
hours of |sum of SHAP over the group|, M27). SHAP describes the fitted booster on these hours; it
is not a causal effect of a feature on the plant.

In [9]:
att = pd.read_csv(ND / "explain" / "tft" / "tft_attention_summary.csv")
cols = ["step", "target", "encoder_positions", "encoder_mass_mean", "same_hour_ratio",
        "argmax_lag_h", "first_position_ratio"]
att[att["block"] == "all"][cols].round(2)

,step,target,encoder_positions,encoder_mass_mean,same_hour_ratio,argmax_lag_h,first_position_ratio
0,tft96,indirect,96,0.22,0.95,-84,1.15
4,tft96,direct,96,0.91,0.95,-12,1.08
8,tft168,indirect,168,0.89,0.98,-168,1.28
12,tft168,direct,168,0.91,0.98,-30,1.11
16,tftbase,indirect,168,0.77,0.82,-138,0.49
20,tftbase,direct,168,0.91,1.07,-161,0.94


**What it shows.** The TFT attention of the development models on validation: the share
of the encoder attention on positions a whole number of days before each step's valid hour,
against the share of such positions (ratio 1 = no preference), the strongest lag before
issuance, and the first position of the window against the same hour of the later days.

**What it means.** No model prefers the hour it forecasts: the ratio is
0.82 to 1.07
(5 of 6 pairs below 1). The daily rhythm of
the attention is a preference for fixed clock hours of earlier days: in the direct route the
profile peaks at 12:00 (TFT-96), 18:00 (TFT-168) and
16:00 (TFT baseline) local time, whatever hour is forecast. Every model attends
only inside its own encoder (C-55). Attention is not attribution; no agreement with SHAP is
claimed as validation (M29).

In [10]:
var = pd.read_csv(ND / "explain" / "tft" / "tft_variable_importance.csv")
enc = var[var["part"] == "encoder"].drop_duplicates(["step", "target", "variable"])
enc.sort_values("variable_weight", ascending=False).groupby(["step", "target"]).head(2)[
    ["step", "target", "variable", "variable_weight"]].round(3)

,step,target,variable,variable_weight
93,tft96,direct,sun_elevation,0.458
414,tftbase,indirect,fcst_gfs_kt_poa_ma3,0.312
26,tft96,indirect,fcst_gfs_kt_poa_daymean,0.260
205,tft168,indirect,dni_clearsky,0.250
490,tftbase,direct,time_to_solar_noon,0.244
191,tft168,indirect,irradiance,0.231
20,tft96,indirect,day_length,0.208
417,tftbase,indirect,fcst_gfs_sunsd,0.197
284,tft168,direct,poa_diffuse_clearsky,0.167
293,tft168,direct,is_weekend,0.151


**What it shows.** The two largest variable-selection weights of each TFT encoder, a
measured series and its fill flag added (X48).

**What it means.** The largest is `sun_elevation` in tft96 direct
(0.46); the leaders differ between models and routes. The weights
come from one seed and a short search; the final run gives the values of the report.

In [11]:
hp = pd.concat([pd.read_csv(XD / "explain" / "hparam_importance.csv"),
                pd.read_csv(ND / "explain" / "hparam_importance.csv")])
hp.dropna(subset=["importance_mean"]).groupby(["step", "param"])[
    ["importance_mean", "importance_std"]].mean().round(2)

importance_mean  importance_std
step   param                                                  
bilstm batch_size                         0.18            0.03
       dropout                            0.29            0.06
       hidden                             0.07            0.02
       lr                                 0.18            0.04
       n_layers                           0.28            0.05
tft168 attention_head_size                0.30            0.04
       dropout                            0.17            0.04
       hidden_continuous_size             0.12            0.03
       hidden_size                        0.09            0.04
       learning_rate                      0.22            0.04
       lstm_layers                        0.11            0.04
tft96  attention_head_size                0.05            0.02
       dropout                            0.19            0.04
       hidden_continuous_size             0.28            0.04
       hidden_size                        0.14            0.03
       learning_rate                      0.26            0.05
       lstm_layers                        0.09            0.03
xgb    colsample_bytree                   0.11            0.03
       learning_rate                      0.31            0.04
       max_depth                          0.15            0.03
       min_child_weight                   0.22            0.03
       n_estimators                       0.11            0.03
       reg_alpha                          0.02            0.01
       reg_lambda                         0.01            0.01
       subsample                          0.07            0.02

**What it shows.** fANOVA importance of the searched hyper-parameters, mean and std over
ten seeded evaluators, averaged per model (M30).

**What it means.** For XGB the most important parameter is `learning_rate`
(0.31 on average). 4 of 6
neural studies lie entirely inside TPE's random start-up (7 TFT trials against 10), so
their importances describe a few random trials only.

## 6. The run reports: warnings and the G-1 list

In [12]:
w = pd.concat([pd.read_csv(XD / "report" / "report_warnings.csv"),
               pd.read_csv(ND / "report" / "report_warnings.csv")])
w[["step", "kind", "status", "text"]]

,step,kind,status,text
0,xgb,gap,matched,indirect_D1_seed42: validation RMSE 0.1004 is ...
1,xgb,edge_space,matched,indirect_D1_seed42: best trial on the edge of ...
2,xgb,edge_space,matched,direct_D1_seed42: best trial on the edge of th...
3,xgb,cap,matched,direct_D2_seed42: best iteration 212 at the ca...
4,xgb,edge_space,matched,direct_D2_seed42: best trial on the edge of th...
0,bilstm,edge_space,matched,bilstm: indirect seed 42: best trial on the ed...
1,tft96,edge_space,matched,tft96: indirect seed 42: best trial on the edg...
2,tft96,edge_space,matched,tft96: direct seed 42: best trial on the edge ...
3,tft168,edge_space,matched,tft168: direct seed 42: best trial on the edge...
4,tft168,edge_space,recomputed only,tft168: indirect seed 42: best trial on the ed...


**What it shows.** The automatic warnings of §3.5 in the two development runs, and
whether the report reproduced them from the stored records (X49).

**What it means.** 10 warnings, 9 reproduced exactly;
1 appears only in the recomputation: the W4 step
judged `hidden_continuous_size` by its sampled value, and the report by the value the network
used, at the edge of its effective range (X45). Every warning was read in the reviews of R9-06,
R9-17 and R9-21.

In [13]:
g1 = pd.concat([pd.read_csv(XD / "report" / "report_g1_points.csv"),
                pd.read_csv(ND / "report" / "report_g1_points.csv")])
g1.groupby("point").size().rename("facts").reset_index()

,point,facts
0,best check against the step budget,2
1,best value on the edge of the search space,8
2,final fit against its best trial,1
3,margin against climatology (kW DM),12
4,mean bias on validation,10
5,study inside TPE's random start-up,4
6,trials stopped at their cap,1


**What it shows.** The facts of the two runs that inform the list of gate G-1, grouped by
point (38 facts in all).

**What it means.** The report states facts and gives no verdict. The points stay a list until
G-1; no validation decision is changed because of them without the user (stage-9 plan §19.1).

## 7. The final run

In [14]:
from solarpv.pipeline.runner import scale_dev_to_final

rows = []
for run, step in [(XD, "xgb"), *((ND, s) for s in
                  ("bilstm", "tft96", "tft168", "tftbase", "patchtst"))]:
    folder = run / step
    rec = json.loads((folder / "step.json").read_text(encoding="utf-8"))
    ratio, upper = scale_dev_to_final(step, folder)
    rows.append({"step": step, "dev_h": rec["duration_s"] / 3600, "fits_ratio": ratio,
                 "final_h": rec["duration_s"] * ratio / 3600,
                 "upper_h": rec["duration_s"] * upper / 3600})
est = pd.DataFrame(rows)
print(f"final run: about {est['final_h'].sum():.1f} h, up to {est['upper_h'].sum():.1f} h")
est.round(2)

final run: about 8.6 h, up to 19.0 h


,step,dev_h,fits_ratio,final_h,upper_h
0,xgb,0.09,9.00,0.80,4.14
1,bilstm,0.16,3.00,0.48,0.49
2,tft96,1.20,2.62,3.16,6.39
3,tft168,1.44,2.62,3.78,7.21
4,tftbase,0.19,1.00,0.19,0.58
5,patchtst,0.22,1.00,0.22,0.22


**What it shows.** The duration of each model step of the final run, scaled from its
development run by the number of fits of the two budgets (X43), and its upper bound if every
neural fit ran to its epoch cap and every XGB trial took as long as the longest dev trial.

**What it means.** The total and the longest step decide whether the final run goes in one
sequence or in segments (`docs/runbook_final.md`, after the college IT answers Q-15). The run has
two phases (D-32): training, validation and the stage-10 acceptance first; the test split only
afterwards, by the step `test`, with every read logged.

**Reproduce.** The two run commands at the top of `scripts/make_models_notebook.py`, then
`python scripts/make_models_notebook.py`.